![LaunchMind Banner](launchmind_banner.png)

# 🚀 Masterpiece Edition: LaunchMind MAS

This version is optimized for **Maximum Free Tier Efficiency**. Due to potential quota limits on Gemini, we utilize **Groq** as the primary engine for high-frequency reasoning while keeping Gemini as a fallback.

In [ ]:
import os, json, uuid, time, base64, requests
from datetime import datetime
from typing import Dict, List, Optional
from dotenv import load_dotenv

import google.generativeai as genai
try:
    from groq import Groq
    import resend
except ImportError:
    !pip install groq resend google-generativeai python-dotenv sendgrid slack_sdk
    from groq import Groq
    import resend
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

load_dotenv()
MOCK_MODE = False

In [ ]:
class MessageBus:
    def __init__(self):
        self.bus: Dict[str, List[dict]] = {}
        self.history: List[dict] = []
    def send_message(self, from_agent, to_agent, m_type, payload, parent_id=None):
        msg = {"message_id": str(uuid.uuid4()), "from_agent": from_agent, "to_agent": to_agent, "message_type": m_type, "payload": payload, "timestamp": datetime.now().isoformat() + "Z", "parent_message_id": parent_id}
        if to_agent not in self.bus: self.bus[to_agent] = []
        self.bus[to_agent].append(msg); self.history.append(msg)
        print(f"[📩] {from_agent.upper()} ➔ {to_agent.upper()} ({m_type.upper()})")
        return msg["message_id"]
    def get_messages(self, agent_name):
        msgs = self.bus.get(agent_name, [])
        self.bus[agent_name] = []; return msgs

In [ ]:
class LLMHandler:
    def __init__(self, provider="groq", mock=False):
        self.mock, self.provider, self.gemini_key, self.groq_key = mock, provider, os.getenv("GEMINI_API_KEY"), os.getenv("GROQ_API_KEY")
        if not mock:
            if self.groq_key: self.groq_client = Groq(api_key=self.groq_key)
            elif self.gemini_key: genai.configure(api_key=self.gemini_key); self.model = genai.GenerativeModel('gemini-2.0-flash')
    def call(self, sys, usr, json_m=False, retries=3):
        if self.mock: return '{"mock": "data"}' if json_m else "PASS"
        for attempt in range(retries + 1):
            try:
                if self.groq_key:
                    full_sys = sys + " Respond in JSON format." if json_m and "json" not in sys.lower() else sys
                    res = self.groq_client.chat.completions.create(messages=[{"role": "system", "content": full_sys}, {"role": "user", "content": usr}], model="llama-3.3-70b-versatile", response_format={"type": "json_object" if json_m else "text"})
                    resp = res.choices[0].message.content
                elif self.gemini_key:
                    prompt = f"{sys}\n\n{usr}"
                    if json_m: prompt += "\n\nRespond ONLY with JSON."
                    resp = self.model.generate_content(prompt).text.strip()
                if json_m:
                    text = resp.strip()
                    if text.startswith("```json"): text = text.split("```json")[1].split("```")[0].strip()
                    elif text.startswith("```"): text = text.split("```")[1].split("```")[0].strip()
                    return text
                return resp
            except Exception as e:
                if attempt == retries: raise e
                time.sleep(2)
        raise ValueError("No API Key.")

In [ ]:
class CEOAgent:
    def __init__(self, bus, mock=False): self.name, self.bus, self.llm = "ceo", bus, LLMHandler(provider="groq", mock=mock)
    def start_process(self, idea):
        print(f"💼 CEO STARTING: {idea}")
        tasks = json.loads(self.llm.call("Split startup idea into JSON tasks for: product, engineer, marketing.", idea, json_m=True))
        self.bus.send_message(self.name, "product", "task", {"idea": idea, "focus": tasks.get("product", "Full spec")})
    def review_output(self, msg):
        rev = self.llm.call("Review result. Be critical. FAIL if bad.", str(msg["payload"]))
        if "FAIL" in rev.upper(): 
            self.bus.send_message(self.name, msg["from_agent"], "revision_request", {"feedback": rev})
            return False
        print(f"✅ {msg['from_agent'].upper()} passed review."); return True

class ProductAgent:
    def __init__(self, bus, mock=False): self.name, self.bus, self.llm = "product", bus, LLMHandler(provider="groq", mock=mock)
    def process(self):
        for msg in self.bus.get_messages(self.name):
            if msg["message_type"] in ["task", "revision_request"]:
                spec = json.loads(self.llm.call("Generate Spec JSON: value_proposition, personas, features (ranked 1-5), user_stories.", str(msg["payload"]), json_m=True))
                for a in ["engineer", "marketing"]: self.bus.send_message(self.name, a, "task", {"spec": spec})
                self.bus.send_message(self.name, "ceo", "confirmation", {"status": "Ready"})

class EngineerAgent:
    def __init__(self, bus, mock=False): self.name, self.bus, self.llm = "engineer", bus, LLMHandler(provider="groq", mock=mock)
    def process(self):
        for msg in self.bus.get_messages(self.name):
            if msg["message_type"] == "task":
                html = self.llm.call("Generate Landing HTML/CSS.", str(msg["payload"]["spec"]))
                self.bus.send_message(self.name, "ceo", "result", {"status":"Skipped", "html": html})

class MarketingAgent:
    def __init__(self, bus, mock=False): self.name, self.bus, self.llm = "marketing", bus, LLMHandler(provider="groq", mock=mock)
    def process(self):
        for msg in self.bus.get_messages(self.name):
            if msg["message_type"] == "task":
                self.copy = json.loads(self.llm.call("Generate assets JSON: tagline, desc, email, posts.", str(msg["payload"]["spec"]), json_m=True))
                self.bus.send_message(self.name, "ceo", "confirmation", {"status": "Ready"})
            elif msg["message_type"] == "pr_info":
                self.bus.send_message(self.name, "ceo", "result", {"copy": self.copy})

class QAAgent:
    def __init__(self, bus, mock=False): self.name, self.bus, self.llm = "qa", bus, LLMHandler(provider="groq", mock=mock)
    def process(self):
        for msg in self.bus.get_messages(self.name):
            if msg["message_type"] == "task":
                rev = self.llm.call("Review vs Spec. Verdict PASS/FAIL?", str(msg["payload"]))
                self.bus.send_message(self.name, "ceo", "result", {"verdict": "PASS" if "PASS" in rev.upper() else "FAIL", "rev": rev})

def run_sim(idea, mock=False):
    bus = MessageBus()
    ceo, product, engineer, marketing, qa = CEOAgent(bus, mock), ProductAgent(bus, mock), EngineerAgent(bus, mock), MarketingAgent(bus, mock), QAAgent(bus, mock)
    st = {"eng": None, "mkt": None}
    ceo.start_process(idea)
    for turn in range(15):
        product.process(); engineer.process(); marketing.process(); qa.process()
        for m in bus.get_messages("ceo"):
            if m["message_type"] == "result":
                if ceo.review_output(m):
                    if m["from_agent"] == "engineer":
                        st["eng"] = m["payload"]; bus.send_message("ceo", "marketing", "pr_info", {"pr_url": "Skipped"})
                    elif m["from_agent"] == "marketing": st["mkt"] = m["payload"]
                    elif m["from_agent"] == "qa" and m["payload"].get("verdict") == "PASS":
                        print("\n🎊 SUCCESS!"); return
        if st["eng"] and st["mkt"] and not [h for h in bus.history if h["from_agent"] == "qa"]:
            spec = next((h["payload"]["spec"] for h in bus.history if h["from_agent"] == "product"), None)
            if spec: bus.send_message("ceo", "qa", "task", {"engineer_output": st["eng"], "marketing_output": st["mkt"], "spec": spec})
        time.sleep(0.5)
run_sim("A solar-powered smart water bottle.", mock=MOCK_MODE)